## Imports

In [ ]:
import os
import sys

# Allow notebooks to import from scripts/
REPO_ROOT = os.path.abspath("..")
sys.path.insert(0, os.path.join(REPO_ROOT, "scripts"))

import nest_asyncio
nest_asyncio.apply()

from pipeline import MMRAGPipeline

## Configurations

In [ ]:
VLM_MODEL_ID   = "Qwen/Qwen3-VL-8B-Instruct"
VLM_CHECKPOINT = os.path.join(REPO_ROOT, "Qwen3-VL-8B-Instruct")

QUESTION_PATHS = {
    "FinancialReport": os.path.join(REPO_ROOT, "data/raw/REAL-MM-RAG_FinReport/test/qas.jsonl"),
    "TechReport":      os.path.join(REPO_ROOT, "data/raw/REAL-MM-RAG_TechReport/test/qas.jsonl"),
    "TechSlides":      os.path.join(REPO_ROOT, "data/raw/REAL-MM-RAG_TechSlides/test/qas.jsonl"),
}

OCR_EMBEDDINGS_DIR = os.path.join(REPO_ROOT, "scripts", "embeddings_ocr")
RESULTS_DIR        = os.path.join(REPO_ROOT, "experiments", "results", "ocr_rag")

TOP_K    = 3
SAMPLING = {"max_new_tokens": 4096}

## Initialize Pipeline

In [ ]:
pipeline = MMRAGPipeline(
    mode="ocr_rag",
    vlm_model_id=VLM_MODEL_ID,
    vlm_checkpoint_path=VLM_CHECKPOINT,
    ocr_retriever_persist_dir=OCR_EMBEDDINGS_DIR,
    top_k=TOP_K,
)

## Ingest Datasets

Index all three test splits into ChromaDB using OCR-extracted text. This only needs to run once — embeddings are persisted to disk and reloaded automatically on subsequent runs.

In [ ]:
for name, qas_path in QUESTION_PATHS.items():
    print(f"\nIngesting {name}...")
    pipeline.ingest(qas_path)

print("\n✅ All datasets ingested.")

## Inference

In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)

for name, qas_path in QUESTION_PATHS.items():
    output_path = os.path.join(RESULTS_DIR, f"ocr_rag_{name}.json")
    print(f"\n{'='*60}")
    print(f"Running inference: {name}")
    print(f"{'='*60}")
    pipeline.run_inference(
        qas_jsonl_path=qas_path,
        output_path=output_path,
        sampling_params=SAMPLING,
    )

print("\n✅ Inference complete. Results saved to:", RESULTS_DIR)

## Evaluation

In [ ]:
# Run evaluate.py from the repo root to compute BLEU, ROUGE, BERTScore, and LLM-as-judge:
#
#   python scripts/evaluate.py experiments/results/ocr_rag/*.json \
#       --output experiments/results/ocr_rag/metrics_summary.json
#
# Set OPENAI_API_KEY in your environment to also enable LLM-as-judge scoring.